In [ ]:
# -*- coding: utf-8 -*-
"""
### Colab 크롤러 v7 - 대규모 데이터 수집 & Google Drive 자동 저장

**기능:**
- 12개 국가 여성 셀카 사진 각각 최소 300장 수집
- Google Drive에 자동 저장
- 진행 상황 자동 저장 및 복구
- 중복 이미지 URL 다운로드 방지
"""

# 1. 패키지 설치 및 환경 설정
def install_requirements():
    """필요한 패키지 자동 설치"""
    print("📦 필요한 패키지 설치 중...")

    packages = [
        "selenium==4.15.2",
        "pillow==10.1.0",
        "requests==2.31.0",
        "webdriver-manager==4.0.1"
    ]

    for package in packages:
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", package],
                         check=True, capture_output=True)
            print(f"✅ {package} 설치 완료")
        except subprocess.CalledProcessError as e:
            print(f"❌ {package} 설치 실패: {e}")

    # Chrome 드라이버 설치
    try:
        subprocess.run(["apt-get", "update"], check=True, capture_output=True)
        subprocess.run(["apt-get", "install", "-y", "chromium-chromedriver"],
                      check=True, capture_output=True)
        print("✅ Chrome 드라이버 설치 완료")
    except subprocess.CalledProcessError as e:
        print(f"❌ Chrome 드라이버 설치 실패: {e}")

def setup_environment():
    """환경 설정"""
    import os
    os.environ['PATH'] += ':/usr/bin/chromedriver'

    try:
        import torch
        if torch.cuda.is_available():
            print("🚀 GPU 가속 사용 가능")
        else:
            print("⚠️ GPU 사용 불가 - CPU 모드로 실행")
    except ImportError:
        print("⚠️ PyTorch 미설치 - CPU 모드로 실행")

# 패키지 설치 및 환경 설정 실행
import sys
import subprocess
install_requirements()
setup_environment()


In [ ]:
# 2. Google Drive 마운트
from google.colab import drive

def setup_google_drive():
    """Google Drive 마운트"""
    print("🔗 Google Drive 연결 중...")
    try:
        drive.mount('/content/drive')
        print("✅ Google Drive 연결 완료")
        return True
    except Exception as e:
        print(f"❌ Google Drive 연결 실패: {e}")
        return False

# Google Drive 연결 실행
setup_google_drive()



In [ ]:
# 3. 라이브러리 임포트
import os
import sys
import subprocess
import time
import requests
import json
import re
import zipfile
import shutil
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from PIL import Image
import io



In [ ]:
# 4. 크롤러 클래스 정의
class LargeScaleColabCrawler:
    def __init__(self):
        """크롤러 초기화"""
        self.driver = None
        self.setup_driver()

    def setup_driver(self):
        """Chrome 드라이버 설정 - 안정성 최적화"""
        chrome_options = Options()
        chrome_options.add_argument('--headless')
        chrome_options.add_argument('--no-sandbox')
        chrome_options.add_argument('--disable-dev-shm-usage')
        chrome_options.add_argument('--disable-gpu')
        chrome_options.add_argument('--window-size=1920,1080')
        chrome_options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
        chrome_options.add_argument('--disable-blink-features=AutomationControlled')
        chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
        chrome_options.add_experimental_option('useAutomationExtension', False)

        # 안정성을 위한 추가 옵션
        chrome_options.add_argument('--disable-extensions')
        chrome_options.add_argument('--disable-plugins')
        chrome_options.add_argument('--disable-images')
        chrome_options.add_argument('--disable-javascript')
        chrome_options.add_argument('--disable-web-security')
        chrome_options.add_argument('--allow-running-insecure-content')
        chrome_options.add_argument('--disable-features=VizDisplayCompositor')

        try:
            self.driver = webdriver.Chrome(options=chrome_options)
            self.driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
            self.driver.set_page_load_timeout(30)
            self.driver.implicitly_wait(10)
            print("✅ Chrome 드라이버 초기화 완료 (안정성 최적화)")
        except Exception as e:
            print(f"❌ 드라이버 초기화 실패: {e}")
            raise

    def scroll_page(self, max_scrolls=15):
        """페이지 스크롤하여 더 많은 이미지 로드"""
        for i in range(max_scrolls):
            self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            print(f"📜 스크롤 {i+1}/{max_scrolls}")

    def find_image_elements(self):
        """다양한 CSS 선택자로 이미지 요소 찾기"""
        selectors = [
            "img.rg_i",  # 기존 선택자
            "img[data-src]",  # 지연 로딩 이미지
            "img[src*='http']",  # HTTP 이미지
            ".rg_i",  # 이미지 컨테이너
            "img[alt*='celebrity']",  # 셀럽 이미지
            "img[alt*='profile']",  # 프로필 이미지
            "img[alt*='actor']",  # 배우 이미지
            "img[alt*='actress']",  # 여배우 이미지
            "img[alt*='star']",  # 스타 이미지
            "img[alt*='famous']",  # 유명인 이미지
            "img[alt*='portrait']",  # 초상화
            "img[alt*='face']",  # 얼굴
            "img[alt*='woman']",  # 여성
            "img[alt*='female']",  # 여성
            "img[alt*='influencer']",  # 인플루언서
            "img[alt*='model']",  # 모델
        ]

        all_elements = []
        for selector in selectors:
            try:
                elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                if elements:
                    print(f"✅ {selector}로 {len(elements)}개 이미지 요소 발견")
                    all_elements.extend(elements)
            except Exception as e:
                print(f"⚠️ {selector} 선택자 실패: {e}")
                continue

        # 중복 제거
        unique_elements = []
        seen_urls = set()
        for element in all_elements:
            try:
                url = element.get_attribute('src')
                if url and url not in seen_urls:
                    unique_elements.append(element)
                    seen_urls.add(url)
            except:
                continue

        print(f"✅ 총 {len(unique_elements)}개 고유 이미지 요소 발견")
        return unique_elements

    def extract_image_url(self, img_element):
        """원본 이미지 URL 추출 - 완전히 새로운 접근 방식"""
        try:
            # 썸네일 URL 확인
            thumbnail_url = img_element.get_attribute('src')

            # 잘못된 URL 필터링
            if not thumbnail_url or not thumbnail_url.startswith('http'):
                return None

            # Google 썸네일 URL 처리 - 원본 URL 추출
            if 'gstatic.com' in thumbnail_url and 'encrypted-tbn' in thumbnail_url:
                # 방법 1: 부모 링크에서 원본 URL 추출
                try:
                    parent_link = img_element.find_element(By.XPATH, "./..")
                    if parent_link.tag_name == 'a':
                        original_url = parent_link.get_attribute('href')
                        if original_url and 'http' in original_url and not original_url.endswith('.gif'):
                            print(f"✅ 원본 URL 추출 (부모 링크): {original_url[:50]}...")
                            return original_url
                except:
                    pass

                # 방법 2: JavaScript로 원본 URL 추출
                try:
                    original_url = self.driver.execute_script("""
                        var img = arguments[0];
                        var parent = img.closest('a');
                        if (parent && parent.href) {
                            return parent.href;
                        }
                        return null;
                    """, img_element)

                    if original_url and 'http' in original_url and not original_url.endswith('.gif'):
                        print(f"✅ 원본 URL 추출 (JavaScript): {original_url[:50]}...")
                        return original_url
                except:
                    pass

                # 방법 3: 이미지 클릭으로 원본 URL 추출 (최적화)
                try:
                    # JavaScript로 클릭
                    self.driver.execute_script("arguments[0].click();", img_element)
                    time.sleep(3)  # 대기 시간 최적화

                    # 현재 페이지의 URL 확인
                    current_url = self.driver.current_url
                    if 'imgres' in current_url or 'search' in current_url:
                        # 이미지 뷰어 페이지에서 큰 이미지 찾기 (빠른 선택자만)
                        large_img_selectors = [
                            "img.r48jcc",  # Google 이미지 뷰어의 큰 이미지
                            "img[src*='http']",  # HTTP 이미지
                            ".n3VNCb",  # Google 이미지 클래스
                            "img[src*='usercontent']",  # 사용자 콘텐츠
                        ]

                        for selector in large_img_selectors:
                            try:
                                large_imgs = self.driver.find_elements(By.CSS_SELECTOR, selector)
                                for large_img in large_imgs:
                                    url = large_img.get_attribute('src')
                                    if url and url.startswith('http') and not url.endswith('.gif'):
                                        if url != thumbnail_url and 'gstatic.com' not in url:
                                            print(f"✅ 클릭 후 원본 URL 발견: {url[:50]}...")
                                            return url
                            except Exception as e:
                                continue

                        # 페이지에서 직접 이미지 URL 추출 시도 (빠른 패턴만)
                        try:
                            page_source = self.driver.page_source
                            # 이미지 URL 패턴 찾기 (최적화된 패턴)
                            import re
                            img_patterns = [
                                r'https://[^"\\s]+\\.(?:jpg|jpeg|png|webp)',
                                r'https://[^"\\s]+/images[^"\\s]+',
                            ]

                            for pattern in img_patterns:
                                matches = re.findall(pattern, page_source)
                                for match in matches:
                                    if match != thumbnail_url and 'gstatic.com' not in match:
                                        print(f"✅ 페이지에서 원본 URL 발견: {match[:50]}...")
                                        return match
                        except:
                            pass

                except Exception as e:
                    print(f"⚠️ 이미지 클릭 실패: {e}")

                # 썸네일 URL 자체 사용 (마지막 수단)
                print(f"⚠️ Google 썸네일 URL 사용: {thumbnail_url[:50]}...")
                return thumbnail_url

            # 일반 이미지 URL 처리
            if thumbnail_url.startswith('http') and not thumbnail_url.endswith('.gif'):
                # 이미지 확장자 확인
                valid_extensions = ['.jpg', '.jpeg', '.png', '.webp']
                if any(ext in thumbnail_url.lower() for ext in valid_extensions):
                    print(f"✅ 직접 이미지 URL: {thumbnail_url[:50]}...")
                    return thumbnail_url

            # 썸네일 URL 사용 (마지막 수단)
            if thumbnail_url and thumbnail_url.startswith('http'):
                print(f"⚠️ 썸네일 URL 사용: {thumbnail_url[:50]}...")
                return thumbnail_url

            return None

        except Exception as e:
            print(f"⚠️ 이미지 URL 추출 실패: {e}")
            return None

    def download_image(self, url, save_path):
        """고화질 이미지 다운로드 및 검증 - 완전히 개선된 버전"""
        try:
            # 잘못된 URL 필터링
            if not url or not url.startswith('http'):
                print(f"⚠️ 잘못된 URL 형식: {url}")
                return False

            # Google 썸네일 URL 필터링 (원본 이미지만 다운로드)
            if 'gstatic.com' in url and 'encrypted-tbn' in url:
                print(f"⚠️ Google 썸네일 URL 제외 (원본 이미지 필요): {url[:50]}...")
                return False

            if any(bad in url.lower() for bad in ['fonts.gstatic.com', 'productlogos', 'favicon', 'logo']):
                print(f"⚠️ 잘못된 이미지 URL 제외: {url[:50]}...")
                return False

            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
                'Referer': 'https://www.google.com/',
                'Accept': 'image/webp,image/apng,image/*,*/*;q=0.8',
                'Accept-Encoding': 'gzip, deflate, br',
                'Accept-Language': 'en-US,en;q=0.9',
                'Cache-Control': 'no-cache',
                'Pragma': 'no-cache'
            }

            response = requests.get(url, headers=headers, timeout=30)  # 타임아웃 증가
            response.raise_for_status()

            # Content-Type 확인
            content_type = response.headers.get('content-type', '').lower()
            if not content_type.startswith('image/'):
                print(f"⚠️ 이미지가 아닌 파일: {content_type}")
                return False

            # 이미지 검증
            try:
                img = Image.open(io.BytesIO(response.content))
            except Exception as e:
                print(f"⚠️ 이미지 파일이 아님: {e}")
                return False

            # 크기 기준 더욱 완화 (최소 100px로 다시 상향)
            min_size = min(img.width, img.height)
            if min_size < 100:
                print(f"⚠️ 이미지가 너무 작음: {img.width}x{img.height} (최소: 100px)")
                return False

            # 너무 가로세로 비율이 극단적인 이미지 제외 (1:5 이상)
            ratio = max(img.width, img.height) / min(img.width, img.height)
            if ratio > 5:
                print(f"⚠️ 이미지 비율이 너무 극단적: {img.width}x{img.height} (비율: {ratio:.1f})")
                return False

            # 이미지 모드 확인 및 변환
            if img.mode in ('RGBA', 'LA', 'P'):
                # 투명도가 있는 이미지는 RGB로 변환
                if img.mode == 'RGBA':
                    # 흰색 배경에 합성
                    background = Image.new('RGB', img.size, (255, 255, 255))
                    background.paste(img, mask=img.split()[-1] if img.mode == 'RGBA' else None)
                    img = background
                else:
                    img = img.convert('RGB')

            # 고품질로 저장 (quality 파라미터 문제 완전 해결)
            try:
                # quality 파라미터 없이 저장
                img.save(save_path, 'JPEG', optimize=True)
            except Exception as e:
                # JPEG 실패 시 PNG로 시도
                try:
                    save_path_png = save_path.replace('.jpg', '.png')
                    img.save(save_path_png, 'PNG')
                    print(f"✅ PNG로 저장: {save_path_png}")
                    return True
                except Exception as e2:
                    print(f"⚠️ 이미지 저장 실패: {e2}")
                    return False

            print(f"✅ 이미지 저장: {img.width}x{img.height} (최소 크기: {min_size}px)")
            return True

        except requests.exceptions.RequestException as e:
            print(f"⚠️ 네트워크 오류: {url[:50]}... - {e}")
            return False
        except Exception as e:
            print(f"⚠️ 이미지 다운로드 실패: {url[:50]}... - {e}")
            return False

    def get_search_queries(self, country):
        """국가별 검색 쿼리 생성"""
        queries = {
            "british": [
                "british girl selfie", "british woman selfie", "uk girl selfie",
                "uk woman portrait", "british girl casual photo", "british woman candid"
            ],
            "chinese": [
                "中国女生 自拍", "中国女性 自拍", "中国女生 日常 自拍",
                "中国女性 生活照", "中国女生 街拍 自拍", "中国美女 自拍"
            ],
            "ethiopian": [
                "ethiopian girl selfie", "ethiopian woman selfie", "ethiopian girl portrait",
                "ethiopian woman casual photo", "ethiopian beauty selfie", "ethiopian girl candid"
            ],
            "french": [
                "femme française selfie", "fille française selfie", "femme française photo portrait",
                "selfie fille française", "photo de femme française", "photo portrait française"
            ],
            "indian": [
                "indian beauty selfie", "indian woman selfie", "indian girl selfie",
                "indian girl portrait", "indian woman candid", " भारतीय लड़की सेल्फी"
            ],
            "indigenous": [
                "native american woman selfie", "indigenous woman selfie", "first nations woman selfie",
                "native american girl portrait", "indigenous beauty selfie", "native woman casual photo"
            ],
            "japanese": [
                "日本人女性 自撮り", "日本人女性 セルフィー", "日本人 素人 自撮り",
                "日本人女性 普通 自撮り", "日本人女性 日常 自撮り", "日本人女性 カジュアル 写真"
            ],
            "korean": [
                "한국 여자 일반인 셀카", "한국 여성 셀카", "한국 여자 일상 셀카",
                "한국 여성 자연스러운 셀카", "한국 여자 사진", "한국 여성 일상 사진"
            ],
            "mexican": [
                "mujeres mexicanas selfie", "mujer mexicana selfie", "selfie mujer mexicana",
                "mujer mexicana foto casual", "foto retrato mujer mexicana", "mujer mexicana foto natural"
            ],
            "nigerian": [
                "nigerian girl selfie", "nigerian woman selfie", "nigerian beauty selfie",
                "nigerian girl portrait", "nigerian woman casual photo", "nigerian girl candid"
            ],
            "russian": [
                "русская девушка селфи", "русская женщина селфи", "селфи русской девушки",
                "фото русской женщины", "портрет русской девушки", "русская женщина фото"
            ],
            "saudi": [
                "فتاة سعودية سيلفي", "امرأة سعودية سيلفي", "صورة سيلفي سعودية",
                "سعودية فتاة صورة شخصية", "saudi girl selfie", "saudi woman selfie"
            ]
        }
        return queries.get(country, [])

    def search_and_download(self, country, save_dir, target_count=300, start_index=0):
        """이미지 검색 및 다운로드 - 안정성 최적화 버전"""
        search_queries = self.get_search_queries(country)
        total_downloaded = start_index
        query_index = 0
        consecutive_failures = 0
        seen_image_urls = set()

        while total_downloaded < target_count and query_index < len(search_queries):
            search_query = search_queries[query_index]
            query_url = f"https://www.google.com/search?q={search_query}&tbm=isch&hl=en"

            try:
                print(f"🔍 검색 쿼리: {search_query}")
                print(f"📊 현재 수집: {total_downloaded}/{target_count}")
                self.driver.get(query_url)
                time.sleep(3)
                self.scroll_page(max_scrolls=10)
                img_elements = self.find_image_elements()

                if not img_elements:
                    query_index += 1
                    continue

                os.makedirs(save_dir, exist_ok=True)
                successful_downloads = 0

                for img in img_elements:
                    if total_downloaded >= target_count: break
                    try:
                        image_url = self.extract_image_url(img)
                        if image_url:
                            if image_url in seen_image_urls:
                                print(f"⚠️ 중복된 이미지 URL 건너뛰기: {image_url[:50]}...")
                                continue
                            
                            filename = f"{country}_{total_downloaded + 1:04d}.jpg"
                            save_path = os.path.join(save_dir, filename)

                            if self.download_image(image_url, save_path):
                                seen_image_urls.add(image_url)
                                total_downloaded += 1
                                successful_downloads += 1
                                consecutive_failures = 0
                                print(f"✅ 다운로드 완료: {filename}")
                                if total_downloaded % 10 == 0:
                                    # This is a placeholder for notebook-specific save logic
                                    print(f"💾 진행 상황 저장 시점: {country} - {total_downloaded}장 완료")
                            else:
                                consecutive_failures += 1
                        else:
                            consecutive_failures += 1
                        
                        if consecutive_failures >= 20:
                            break 
                        time.sleep(0.5)
                    except Exception as e:
                        print(f"⚠️ 이미지 처리 실패: {e}")
                        continue
                query_index += 1
            except Exception as e:
                print(f"❌ 검색 실패: {e}")
                query_index += 1
                continue
        return total_downloaded

    def save_to_google_drive(self, country, local_path):
        """Google Drive에 저장"""
        try:
            drive_path = f"/content/drive/MyDrive/WhosYourAncestor/data/female/{country}"
            os.makedirs(drive_path, exist_ok=True)
            import shutil
            for file in os.listdir(local_path):
                if file.endswith('.jpg'):
                    src = os.path.join(local_path, file)
                    dst = os.path.join(drive_path, file)
                    shutil.copy2(src, dst)
            print(f"✅ Google Drive에 저장 완료: {drive_path}")
            return True
        except Exception as e:
            print(f"❌ Google Drive 저장 실패: {e}")
            return False

    def close(self):
        """드라이버 종료"""
        if self.driver:
            self.driver.quit()
            print("✅ 드라이버 종료")


